In [1]:
import torch
from torch import nn


device = "cuda" if torch.cuda.is_available() else "cpu"


class SingleTokenMLP(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(paragraph_dim, 2048),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(2048, decoder_hidden_dim)
        )

    def forward(self, x):
        return self.net(x)


class EndToEndInverter(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim, prefix_len, decoder_model):
        super().__init__()
        self.prefix_len = prefix_len
        self.m_parallel_mlps = nn.ModuleList(
            [SingleTokenMLP(paragraph_dim, decoder_hidden_dim) for _ in range(prefix_len)]
        )
        self.decoder = decoder_model

    def _prefix(self, paragraph_embs):
        return torch.stack([mlp(paragraph_embs) for mlp in self.m_parallel_mlps], dim=1)

    @torch.no_grad()
    def generate(self, paragraph_embs, tokenizer, max_new_tokens=128):
        paragraph_embs = paragraph_embs.to(next(self.parameters()).device)
        prefix = self._prefix(paragraph_embs).to(dtype=self.decoder.dtype)
        outputs = self.decoder.generate(
            inputs_embeds=prefix,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
            repetition_penalty=1.2,
        )
        return tokenizer.batch_decode(outputs, skip_special_tokens=True)


In [2]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download
from peft import get_peft_model, LoraConfig, TaskType
import torch.nn.functional as F
import re

# Matches keys like:
#   decoder.base_model.model.model.layers.19.self_attn.q_proj.lora_A.default.weight
LORA_KEY_PATTERN = re.compile(r"\.([a-zA-Z0-9_]+)\.lora_(A|B)\.default\.weight$")


def detect_lora_config(state_dict):
    """
    Recovers target_modules and rank r directly from a checkpoint's state dict,
    instead of assuming a single hardcoded LoraConfig for every checkpoint.

    NOTE: lora_alpha cannot be recovered this way (it only scales the adapter
    output, it is never stored as a tensor). We default alpha = 2 * r, which
    matches the convention used in the training script, but if a given run
    used a different alpha this will silently use the wrong scale. Pass
    lora_alpha explicitly per-config below if you know the true value.
    """
    target_modules = set()
    ranks_seen = set()

    for key, tensor in state_dict.items():
        match = LORA_KEY_PATTERN.search(key)
        if match is None:
            continue
        proj_name, ab = match.group(1), match.group(2)
        target_modules.add(proj_name)
        # lora_A: [r, in_dim] -> r is dim 0 | lora_B: [out_dim, r] -> r is dim 1
        ranks_seen.add(tensor.shape[0] if ab == "A" else tensor.shape[1])

    if not target_modules:
        raise ValueError(
            "No LoRA weights found in this checkpoint (no '*.lora_A/B.default.weight' "
            "keys). Cannot auto-detect a LoRA config for it."
        )
    if len(ranks_seen) > 1:
        raise ValueError(f"Multiple LoRA ranks found in one checkpoint: {sorted(ranks_seen)}")

    return {"target_modules": sorted(target_modules), "r": ranks_seen.pop()}


class InverterFactory:
    def __init__(self):
        self.base_name = "Qwen/Qwen3-0.6B"

    def build_decoder(self, target_modules, r, lora_alpha=None):
        base = AutoModelForCausalLM.from_pretrained(
            self.base_name, device_map={"": 0}, trust_remote_code=True
        )

        alpha = lora_alpha if lora_alpha is not None else 2 * r

        lora = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=r,
            lora_alpha=alpha,
            lora_dropout=0.05,
            target_modules=target_modules,
        )

        print(f"  Building decoder with target_modules={target_modules}, r={r}, alpha={alpha}")
        return get_peft_model(base, lora)

    def load(self, repo, filename, prefix_len, paragraph_dim, lora_alpha=None):
        # Download and inspect the checkpoint FIRST, so the decoder we build
        # actually matches what's inside it, instead of guessing up front.
        state_dict = torch.load(
            hf_hub_download(repo_id=repo, filename=filename),
            map_location="cpu",
        )

        detected = detect_lora_config(state_dict)
        decoder = self.build_decoder(
            target_modules=detected["target_modules"],
            r=detected["r"],
            lora_alpha=lora_alpha,
        )

        model = EndToEndInverter(
            paragraph_dim=paragraph_dim,
            decoder_hidden_dim=decoder.config.hidden_size,
            prefix_len=prefix_len,
            decoder_model=decoder,
        )

        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        print(repo, "missing:", len(missing), "unexpected:", len(unexpected))
        if missing or unexpected:
            print("  missing keys sample:", missing[:3])
            print("  unexpected keys sample:", unexpected[:3])

        return model.to(device).eval()

In [3]:
facts = [
    # --- Original 5 Facts ---
    "Paris is the capital of France and it is known for its rich cultural heritage, world-renowned art museums, historic architecture, and famous landmarks like the Eiffel Tower and the Louvre Museum, which houses thousands of works including the Mona Lisa, making the city one of the most visited tourist destinations in the world and a global center for fashion, cuisine, and intellectual life, attracting millions of visitors every year who come to experience its vibrant street life, historical neighborhoods, iconic cafes, and its lasting influence on art, literature, and global culture throughout history.",
    "The Earth revolves around the Sun once every year in an elliptical orbit, and this motion, along with its tilted axis, is responsible for the changing seasons we experience, influencing variations in temperature, daylight hours, and weather patterns across different regions of the planet throughout the year, while also playing a crucial role in sustaining life by regulating climate systems, supporting ecosystems, and enabling the natural cycles that govern agriculture, biodiversity, and environmental balance.",
    "Water boils at one hundred degrees Celsius under standard atmospheric pressure, and this physical property is widely used in cooking, scientific experiments, and industrial processes, while also serving as a reference point in temperature measurement systems and changing slightly with variations in altitude and pressure, making it an essential concept in thermodynamics and practical applications where precise temperature control and understanding of phase changes are critical for efficiency and safety.",
    "The human brain controls the entire body by sending electrical and chemical signals through the nervous system, allowing us to think, move, feel emotions, process sensory information, store memories, make decisions, and respond effectively to our environment in both conscious and unconscious ways, while also coordinating complex bodily functions such as breathing, heartbeat, and hormonal regulation, making it one of the most sophisticated and vital organs in the human body.",
    "Mount Everest is the highest mountain above sea level, standing at over 8800 meters, and it is located in the Himalayas on the border between Nepal and China, attracting climbers from around the world despite its extreme conditions, including low oxygen levels, freezing temperatures, and challenging terrain, and it remains a symbol of human endurance and exploration as expeditions continue to push the limits of physical and mental capability in one of the harshest environments on Earth.",

    # --- 45 New Facts for Robust Evaluation ---
    "Jupiter is the largest planet in our solar system, functioning as a massive gas giant composed primarily of hydrogen and helium, and is widely recognized for its Great Red Spot, a colossal anticyclonic storm that has been continuously raging for centuries, as well as its extensive system of dozens of moons including Ganymede.",
    "The Amazon Rainforest spans across several South American countries and is considered the most biodiverse terrestrial ecosystem on the planet, serving as a critical global carbon sink that helps regulate the Earth's climate by absorbing massive amounts of carbon dioxide while producing oxygen and maintaining the water cycle.",
    "Deoxyribonucleic acid, commonly known as DNA, is the complex double-helix molecule that carries the genetic instructions responsible for the development, functioning, growth, and reproduction of all known living organisms, utilizing sequences of four fundamental nucleotide bases to encode biological information.",
    "The Great Wall of China is an extensive series of ancient fortifications built across the historical northern borders of China to protect against nomadic invasions, representing one of the most remarkable architectural and military engineering feats in human history and spanning thousands of miles across diverse terrains.",
    "Photosynthesis is the fundamental biological process by which green plants, algae, and certain bacteria convert light energy into chemical energy, utilizing carbon dioxide and water to produce glucose and releasing oxygen as a byproduct, which is absolutely essential for sustaining the vast majority of life on Earth.",
    "Black holes are regions of spacetime exhibiting extremely strong gravitational acceleration where nothing, not even light, can escape their pull, typically forming from the collapsed cores of massive stars at the end of their life cycles and marked by an invisible boundary known as the event horizon.",
    "The Internet is a globally connected network system that uses the standard Internet Protocol suite to serve billions of users worldwide, revolutionizing human communication, commerce, and access to information by linking private, public, academic, business, and government networks through a broad array of electronic and optical networking technologies.",
    "William Shakespeare was an English playwright, poet, and actor who is widely regarded as the greatest writer in the English language and the world's pre-eminent dramatist, having authored masterpieces such as Hamlet, Romeo and Juliet, and Macbeth that continue to be studied and performed globally centuries after his death.",
    "The Sahara Desert is the largest hot desert in the world, covering much of North Africa with a harsh, arid environment characterized by vast sand dunes, rocky plateaus, and extreme temperature fluctuations, yet it historically supported vital trade routes and uniquely adapted ecosystems.",
    "Albert Einstein developed the theory of relativity, one of the two pillars of modern physics, fundamentally altering our understanding of space, time, and gravity, and establishing the world-famous mass-energy equivalence formula which demonstrates that mass can be converted into immense amounts of energy.",
    "The blue whale is a marine mammal belonging to the baleen whale suborder and holds the record as the largest known animal to have ever lived on Earth, possessing a massive heart and sustaining itself primarily by filter-feeding on enormous quantities of tiny crustaceans called krill.",
    "The Colosseum is an elliptical amphitheater in the center of the city of Rome, built of travertine limestone, tuff, and brick-faced concrete, and is renowned for having hosted spectacular gladiatorial contests, public spectacles, and dramatic reenactments of famous battles during the height of the Roman Empire.",
    "Gravity is the fundamental physical force that causes mutual attraction between all things with mass or energy, keeping planets in orbit around stars, holding galaxies together, and giving weight to physical objects on Earth, dictating the macroscopic structure of the entire universe.",
    "The Industrial Revolution was a period of major industrialization and innovation that took place during the late 1700s and early 1800s, transitioning societies from agrarian and handicraft economies to ones dominated by machine manufacturing, steam power, and the establishment of the modern factory system.",
    "Bees are flying insects closely related to wasps and ants, globally celebrated for their role in pollination and for producing honey, playing a structurally vital role in global agriculture and the survival of diverse floral ecosystems by transferring pollen between plants.",
    "The Mariana Trench is the deepest oceanic trench on Earth, located in the western Pacific Ocean, reaching a maximum known depth of nearly eleven thousand meters at the Challenger Deep, and remains one of the most mysterious and unexplored environments on the planet.",
    "Vaccines are biological preparations that provide active acquired immunity to a particular infectious disease, typically containing an agent that resembles a disease-causing microorganism, which stimulates the body's immune system to recognize the agent as a threat, destroy it, and remember it for future protection.",
    "The Renaissance was a fervent period of European cultural, artistic, political, and economic rebirth following the Middle Ages, generally described as taking place from the 14th century to the 17th century, and promoting the rediscovery of classical philosophy, literature, and art.",
    "Octopuses are highly intelligent, soft-bodied cephalopods known for their complex nervous systems, advanced problem-solving abilities, and remarkable capacity for camouflage, utilizing specialized skin cells to instantly change their color, opacity, and texture to blend seamlessly into their marine surroundings.",
    "The Statue of Liberty is a colossal neoclassical copper sculpture on Liberty Island in New York Harbor, gifted to the United States by the people of France in 1886, and widely recognized as a universal symbol of freedom, democracy, and a welcoming sight to immigrants arriving by sea.",
    "Plate tectonics is the scientific theory describing the large-scale motion of the Earth's lithosphere, explaining how the movement of distinct tectonic plates floating on the molten mantle leads to the formation of mountains, oceanic trenches, earthquakes, and volcanic activity along their boundaries.",
    "The Apollo 11 mission in 1969 marked the first time humans successfully landed on the Moon, an extraordinary achievement of engineering and human courage that fulfilled a national goal proposed by President John F. Kennedy and broadcast live to millions of people worldwide.",
    "Trees play a crucial biological and environmental role by absorbing carbon dioxide, providing oxygen, preventing soil erosion, and offering habitats for countless species, making forests essential to maintaining global biodiversity and mitigating the severe impacts of anthropogenic climate change.",
    "The human heart is a muscular organ responsible for continuously pumping oxygenated blood throughout the circulatory system to nourish cells and remove metabolic waste, functioning tirelessly through a complex system of electrical impulses and mechanical valves from before birth until death.",
    "The printing press, invented by Johannes Gutenberg in the 15th century, revolutionized the mass production of books and the dissemination of knowledge, fundamentally accelerating the spread of ideas during the Renaissance, the Scientific Revolution, and the Protestant Reformation.",
    "The Great Barrier Reef is the world's largest coral reef system, located off the coast of Queensland, Australia, and is composed of over two thousand individual reefs that support a staggering diversity of marine life, though it is currently highly vulnerable to the effects of ocean warming.",
    "The Olympic Games are the leading international sporting events featuring summer and winter sports competitions in which thousands of athletes from around the world participate in a variety of competitions, reviving a tradition that originated in ancient Greece to foster global unity.",
    "Electricity is a fundamental form of energy resulting from the existence and movement of charged particles such as electrons, powering virtually all modern technological infrastructure, lighting our cities, and enabling the digital revolution that connects the contemporary world.",
    "The Taj Mahal is an ivory-white marble mausoleum on the right bank of the river Yamuna in the Indian city of Agra, commissioned by the Mughal emperor Shah Jahan to house the tomb of his favorite wife, representing the pinnacle of Mughal architecture and an enduring symbol of love.",
    "Evolution by natural selection, primarily formulated by Charles Darwin, is the scientific process through which populations of living organisms adapt and change over generations, driven by the differential survival and reproduction of individuals due to variations in their phenotypic traits.",
    "The speed of light in a vacuum is exactly 299,792,458 meters per second, serving as a fundamental universal physical constant that dictates the absolute speed limit of the universe and plays a foundational role in the theories of electromagnetism and relativity.",
    "Leonardo da Vinci was a true polymath of the High Renaissance whose areas of interest included invention, painting, sculpting, architecture, science, music, and mathematics, and he is universally recognized for producing enduring masterpieces like the Mona Lisa and The Last Supper.",
    "Antarctica is the Earth's southernmost continent, containing the geographic South Pole and covered almost entirely by an immense ice sheet, representing the coldest, driest, and windiest continent on the planet while holding about seventy percent of the world's fresh water reserves.",
    "The periodic table is a tabular display of the chemical elements, organized by atomic number, electron configuration, and recurring chemical properties, providing a universally understood framework used by scientists to analyze chemical behavior and predict the discovery of new elements.",
    "Artificial intelligence is the simulation of human intelligence processes by machines, especially computer systems, encompassing a wide range of technologies including machine learning, natural language processing, and computer vision to perform tasks that traditionally require human cognitive functions.",
    "Machu Picchu is a 15th-century Inca citadel located high in the Andes Mountains of Peru, renowned for its sophisticated dry-stone construction that fuses massive blocks without mortar, and standing as the most familiar icon of the incredible architectural prowess of the Inca civilization.",
    "The Panama Canal is an artificial waterway in Panama that connects the Atlantic Ocean with the Pacific Ocean, serving as a vital conduit for maritime trade that greatly reduces the time for ships to travel between the two oceans, avoiding the hazardous Cape Horn route.",
    "Antibiotics are a class of antimicrobial substances used extensively in the treatment and prevention of bacterial infections, fundamentally revolutionizing modern medicine and significantly extending human life expectancy since the widespread deployment of penicillin in the 20th century.",
    "The solar system consists of the Sun and the planetary system that orbits it, including eight major planets, their natural satellites, dwarf planets like Pluto, and countless smaller bodies such as asteroids and comets, all bound together by the Sun's immense gravitational field.",
    "Airplanes are powered flying vehicles with fixed wings and a weight greater than that of the air they displace, enabling rapid global transportation of passengers and cargo, fundamentally transforming international commerce, tourism, and military logistics over the last century.",
    "Global warming refers to the long-term heating of Earth's climate system observed since the pre-industrial period, driven primarily by human activities such as the burning of fossil fuels, which drastically increases heat-trapping greenhouse gas levels in the Earth's atmosphere.",
    "The Nile is a major north-flowing river in northeastern Africa and is generally regarded as the longest river in the world, having historically provided the crucial water and fertile soil necessary for the development, survival, and flourishing of ancient Egyptian civilization.",
    "Nuclear energy is the energy in the core of an atom, released either through nuclear fusion or nuclear fission, and it represents a powerful source of low-carbon electricity generation, though it requires meticulous engineering to safely manage the highly radioactive byproducts.",
    "The Milky Way is the galaxy that contains our Solar System, taking the form of a barred spiral galaxy with a diameter of over one hundred thousand light-years, housing billions of stars, massive dust clouds, and a supermassive black hole positioned at its galactic center.",
    "Democracy is a system of government in which state power is vested in the people or the general population of a state, typically functioning through a system of elected representation, the protection of civil liberties, and the adherence to the rule of law and human rights.",
    "Fossils are the preserved remains, impressions, or traces of any once-living thing from a past geological age, providing crucial primary evidence for paleontologists studying the evolutionary history of life on Earth and the historical context of ancient biological ecosystems.",
    "The International Space Station is the largest modular space station currently in low Earth orbit, serving as a microgravity and space environment research laboratory where scientific research is conducted in astrobiology, astronomy, meteorology, and physics by a multinational crew.",
    "Volcanoes are ruptures in the crust of a planetary-mass object, such as Earth, that allow hot lava, volcanic ash, and gases to escape from a magma chamber below the surface, violently shaping landscapes and influencing the atmospheric climate over geological timescales.",
    "The Roman Empire was the post-Republican period of ancient Rome, characterized by an autocratic form of government and vast territorial holdings around the Mediterranean Sea in Europe, North Africa, and Western Asia, leaving a profound legacy on global law, architecture, and language.",
    "Quantum mechanics is a fundamental theory in physics that provides a description of the physical properties of nature at the scale of atoms and subatomic particles, introducing counterintuitive principles such as wave-particle duality and the uncertainty principle."
]

In [4]:
!uv pip install rouge_score nltk absl-py evaluate

Using Python 3.12.13 environment at: /usr
Resolved 50 packages in 658ms                                        
Prepared 1 package in 34ms                                               
Installed 1 package in 6ms                                  
 + evaluate==0.4.6


In [5]:
class Encoder:
    def __init__(self, model_name="Qwen/Qwen3-Embedding-0.6B"):
        self.model = AutoModel.from_pretrained(model_name).to(device).eval()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    def encode(self, texts):
        inputs = self.tokenizer(
            texts, return_tensors="pt", padding=True, truncation=True, max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = self.model(**inputs)

        token_embeddings = outputs.last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1).float()
        sentence_embeddings = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)
        return F.normalize(sentence_embeddings, p=2, dim=1)

In [6]:
import evaluate
import numpy as np
import torch.nn.functional as F

class Evaluator:
    def __init__(self, encoder):
        print("Loading evaluation metrics (this might download some small models)...")
        self.rouge = evaluate.load('rouge')
        self.bleu = evaluate.load('bleu')
        self.meteor = evaluate.load('meteor')
        self.bertscore = evaluate.load('bertscore')
        self.encoder = encoder  # <-- We now give the evaluator access to the encoder!

    def score(self, inputs, outputs):
        # 1. Standard Metrics
        r_score = self.rouge.compute(predictions=outputs, references=inputs)
        b_score = self.bleu.compute(predictions=outputs, references=inputs)
        m_score = self.meteor.compute(predictions=outputs, references=inputs)
        bs_score = self.bertscore.compute(predictions=outputs, references=inputs, lang="en")
        
        # 2. Embedding Cosine Similarity
        # Encode both the original text and the generated text
        input_embs = self.encoder.encode(inputs)
        output_embs = self.encoder.encode(outputs)
        
        # Calculate how mathematically identical they are (1.0 is perfect)
        cos_sims = F.cosine_similarity(input_embs, output_embs, dim=1).cpu().numpy()
        
        return {
            "ROUGE-1": r_score['rouge1'],
            "ROUGE-2": r_score['rouge2'],
            "ROUGE-L": r_score['rougeL'],
            "BLEU": b_score['bleu'],
            "METEOR": m_score['meteor'],
            "BERTScore-F1": np.mean(bs_score['f1']),
            "Cosine-Similarity": np.mean(cos_sims)  
        }

In [7]:
!pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.7 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23

In [8]:
import re

In [15]:
class Runner:
    def __init__(self):
        self.encoder = Encoder()
        self.factory = InverterFactory()
        self.evaluator = Evaluator(self.encoder) # <-- Passed the encoder here!

    def run(self, configs, texts):
        embeddings = self.encoder.encode(texts)
        tokenizer = self.encoder.tokenizer

        results = {}

        for cfg in configs:
            model = self.factory.load(
                repo=cfg["repo"],
                filename=cfg["filename"],
                prefix_len=cfg["prefix_len"],
                paragraph_dim=embeddings.shape[1],
            )

            outputs = model.generate(embeddings, tokenizer, max_new_tokens=128)

            #outputs = [re.sub(r'[^\x00-\x7F]+', '', text) for text in outputs]
            
            scores = self.evaluator.score(texts, outputs)

            unique_name = f"{cfg['repo']} ({cfg['filename']})"
            results[unique_name] = {"scores": scores, "output": outputs}

            del model
            torch.cuda.empty_cache()

        return results

    
    def report(self, texts, results):
        names = list(results.keys())
    
        for i in range(len(texts)):
            print(f"\n{'='*80}")
            print(f"=== Sample {i+1} ===")
            print("ORIGINAL Input:", texts[i])
    
            for n in names:
                print(f"\n--- Model: {n} ---")
                print("Generated Output:", results[n]["output"][i])
                print("\nMetrics for this sample:")
                
                # Calculate scores for just this single sample
                single_score = self.evaluator.score([texts[i]], [results[n]["output"][i]])
                for k, v in single_score.items():
                    val = v.item() if hasattr(v, "item") else v
                    print(f"  {k}: {val:.4f}")
                    
        # Print Overall Averages
        print(f"\n{'='*80}")
        print("OVERALL AVERAGE SCORES (Across all samples)")
        print(f"{'='*80}")
        for n in names:
            print(f"\nModel: {n}")
            for k, v in results[n]["scores"].items():
                val = v.item() if hasattr(v, "item") else v
                print(f"  Avg {k}: {val:.4f}")
   



In [16]:
configs = [
    {
        "repo": "jg-eno/ReLoDer_v1",
        "prefix_len": 64,
        "filename": "best_checkpoint.pt"
    },
    {
        "repo": "jg-eno/ReLoDer_v2",
        "prefix_len": 64,
        "filename": "best_epoch_checkpoint.pt"
    },
     {
        "repo": "jg-eno/ReLoDer_v3",
        "prefix_len": 128,
        "filename": "best_epoch_checkpoint.pt"
    },
     {
        "repo": "jg-eno/ReLoDer_v3",
        "prefix_len": 128,
        "filename": "best_steps_checkpoint.pt"
    },
]

# 1. Create the Runner (This triggers the "Loading evaluation metrics..." print)
runner = Runner()

# 2. Run the models and get the results
results = runner.run(configs, facts)

# 3. Print the beautiful report!
runner.report(facts, results)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading evaluation metrics (this might download some small models)...


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  Building decoder with target_modules=['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj'], r=32, alpha=64
jg-eno/ReLoDer_v3 missing: 0 unexpected: 0


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  Building decoder with target_modules=['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj'], r=32, alpha=64
jg-eno/ReLoDer_v3 missing: 0 unexpected: 0

=== Sample 1 ===
ORIGINAL Input: Paris is the capital of France and it is known for its rich cultural heritage, world-renowned art museums, historic architecture, and famous landmarks like the Eiffel Tower and the Louvre Museum, which houses thousands of works including the Mona Lisa, making the city one of the most visited tourist destinations in the world and a global center for fashion, cuisine, and intellectual life, attracting millions of visitors every year who come to experience its vibrant street life, historical neighborhoods, iconic cafes, and its lasting influence on art, literature, and global culture throughout history.

--- Model: jg-eno/ReLoDer_v1 (best_checkpoint.pt) ---
Generated Output: Paris is known for its rich history, art and literature, music, fashion, architecture, tourism and culture. T